# Lab Day 1 — Neural network training experiments (Forest CoverType)

Notebook tự clone repo, đọc danh sách thí nghiệm từ `code/experiments.yaml`, chuẩn bị thư mục `submission_<MSSV>/`, chạy toàn bộ Part 0–4 và ghi
`figures/`, `results/`, `experiments.xlsx`, `predictions_eval.csv`, `eval_result.json`.

**Cách dùng:** sửa ô **CONFIG** bên dưới (ít nhất là `MSSV`) → *Runtime → Run all* (Colab) / *Restart & Run All*.
Trên Colab nên chọn GPU (*Runtime → Change runtime type → T4 GPU*). Các ô **Nhận xét / Dự đoán** là phần tự viết.

In [ ]:
# ===================== CONFIG — chỉnh ở đây rồi Run All =====================
MSSV          = "XXXXXXXX"            # mã số sinh viên -> thư mục submission_<MSSV>/
REPO_URL      = "https://github.com/Le-Anh-Duy/K4-Track4-Day1-Neuralnetwork.git"
BRANCH        = "main"
CLONE_PARENT  = None                  # None: /content (Colab), /kaggle/working (Kaggle), thư mục hiện tại (máy local)
USE_DRIVE     = False                 # Colab: đặt submission_<MSSV>/ trên Google Drive (an toàn khi bị ngắt kết nối)
DRIVE_DIR     = "/content/drive/MyDrive/K4-Track4-Day1"

CONFIG_FILE   = "experiments.yaml"    # danh sách thí nghiệm (trong code/ của repo)

SKIP_EXISTING = True                  # đã có results/<exp_id>.json (+ checkpoint) cùng cfg -> nạp lại, không chạy lại
QUICK         = False                 # True: chạy thử cả kế hoạch với 2 epoch vào submission_<MSSV>_quick/ (chỉ để kiểm tra pipeline)

In [ ]:
# ===== Clone repo, tạo thư mục nộp, chọn device =====
import os, sys, json, time, shutil, subprocess, math
from pathlib import Path

IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")

def find_repo(start):
    p = Path(start).resolve()
    for q in [p, *p.parents]:
        if (q / "scripts" / "split_data.py").exists() and (q / "data" / "covtype.csv.gz").exists():
            return q
    return None

REPO_ROOT = find_repo(os.getcwd())
if REPO_ROOT is None:
    parent = Path(CLONE_PARENT or ("/content" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else os.getcwd()))
    REPO_ROOT = parent / Path(REPO_URL).stem
    if (REPO_ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
print("REPO_ROOT =", REPO_ROOT)

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SUB_PARENT = Path(DRIVE_DIR)
else:
    SUB_PARENT = REPO_ROOT
SUB_DIR  = SUB_PARENT / (f"submission_{MSSV}" + ("_quick" if QUICK else ""))
CODE_DIR = SUB_DIR / "code"
CODE_DIR.mkdir(parents=True, exist_ok=True)

# Chép code/ của repo vào submission_<MSSV>/code/ (trừ khi notebook đang chạy ngay trong thư mục đó)
if Path(os.getcwd()).resolve() != CODE_DIR.resolve():
    for f in [*(REPO_ROOT / "code").glob("*.py"), *(REPO_ROOT / "code").glob("*.yaml")]:
        shutil.copy2(f, CODE_DIR / f.name)
    if not (CODE_DIR / "lab.ipynb").exists():
        shutil.copy2(REPO_ROOT / "code" / "lab.ipynb", CODE_DIR / "lab.ipynb")
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))

OUT_DIR  = str(SUB_DIR)                                   # = ".." khi đứng trong submission_<MSSV>/code
FIG_DIR, RES_DIR = f"{OUT_DIR}/figures", f"{OUT_DIR}/results"
CKPT_DIR = REPO_ROOT / "checkpoints" / SUB_DIR.name       # trọng số best_state, nằm NGOÀI thư mục nộp
for d in (FIG_DIR, RES_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print("SUB_DIR  =", SUB_DIR)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display, Image
display_img = lambda p: display(Image(p))

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "")
if device == "mps":
    device = "cpu"  # torch.Generator / bincount trên mps còn hạn chế; mạng nhỏ chạy CPU cũng ổn

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval, compute_loss
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
from runner import Runner, load_plan
set_seed(0)

## Part 0 — Dữ liệu
Chia sẵn bằng metadata (`scripts/split_data.py`) → tách validation 20% từ train (phân tầng, seed 42)
→ chuẩn hoá 10 cột số bằng thống kê của phần train còn lại → đưa toàn bộ lên device.

In [ ]:
if not (REPO_ROOT / "data/processed/train.npz").exists():
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True,
                   env={**os.environ, "PYTHONIOENCODING": "utf-8"})

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
assert len(data["X_tr"]) == 371_847 and len(data["X_val"]) == 92_962 and len(data["X_eval"]) == 116_203

dist = pd.DataFrame({name: np.bincount(data[k].cpu().numpy(), minlength=7) for name, k in
                     [("train", "y_tr"), ("val", "y_val"), ("eval", "y_eval")]})
display(pd.concat([dist, (100 * dist / dist.sum()).round(3).add_suffix(" %")], axis=1))

num = data["X_tr"][:, :10]
print("mean 10 cột số trên X_tr:", num.mean(0).cpu().numpy().round(4))
print("std  10 cột số trên X_tr:", num.std(0).cpu().numpy().round(4))
print("mean 10 cột số trên X_val (chỉ để đối chiếu, không dùng để chuẩn hoá):", data["X_val"][:, :10].mean(0).cpu().numpy().round(3))

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
`M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model, khởi tạo He.

In [ ]:
# 1) Model + số tham số
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47_879
print(model)
print("số tham số:", count_params(model))
for hid in EXPECTED_PARAMS:  # kiểm tra luôn M-wide, M-deep
    assert count_params(MLP(hidden=hid)) == EXPECTED_PARAMS[hid], hid

# 2) Shape qua từng lớp với một lô ngẫu nhiên (8, 54)
h = torch.randn(8, 54, device=device)
for layer in model.net:
    h = layer(h)
    print(f"{layer.__class__.__name__:8s} -> {tuple(h.shape)}")
assert model(torch.randn(8, 54, device=device)).shape == (8, 7)

# 3) Loss bước 0 trên val (eval mode)
step0 = evaluate(model, data["X_val"], data["y_val"])
print(f"loss bước 0 trên val = {step0['loss']:.4f}  |  ln 7 = {math.log(7):.4f}  |  acc = {step0['acc']:.4f}")

In [ ]:
# 4) Quá khớp 20 mẫu (không dropout, Adam lr 1e-2, 500 bước) -> loss phải về gần 0
set_seed(1)
tiny = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
idx = torch.randperm(len(data["X_tr"]), device=device)[:20]
xb20, yb20 = data["X_tr"][idx], data["y_tr"][idx]
opt = torch.optim.Adam(tiny.parameters(), lr=1e-2)
losses = []
for step in range(500):
    tiny.train()
    loss = compute_loss(tiny(xb20), yb20, "ce")
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    losses.append(loss.item())
acc20 = (predict(tiny, xb20) == yb20).float().mean().item()
print(f"loss sau 500 bước = {losses[-1]:.2e} | accuracy trên 20 mẫu = {acc20:.2f}")
assert losses[-1] < 1e-2 and acc20 == 1.0

plt.figure(figsize=(6, 3.5))
plt.semilogy(losses); plt.xlabel("bước"); plt.ylabel("CE loss (log)"); plt.title("Quá khớp 20 mẫu (M-base, Adam 1e-2)")
plt.grid(alpha=0.3); plt.show()

# 5) Gradient có chảy tới mọi tham số?
model.train(); model.zero_grad()
compute_loss(model(data["X_tr"][:512]), data["y_tr"][:512], "ce").backward()
for name, p in model.named_parameters():
    print(f"{name:14s} shape {str(tuple(p.shape)):12s} grad norm = {p.grad.norm().item():.4e}")
    assert p.grad is not None and p.grad.norm().item() > 0, name

**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...; gradient: ...

## Part 2 + 3 — Chạy toàn bộ kế hoạch thí nghiệm (một lượt)
Danh sách thí nghiệm nằm trong **`code/experiments.yaml`** (sửa file đó trên repo để thêm/bớt thí nghiệm). Ô dưới chạy theo thứ tự:
1. **Tìm lr baseline** (SGD+momentum, nhiều lr, seed 1) → `BASE_LR` = lr có val macro-F1 cao nhất (chỉ dùng val).
2. **Baseline** với nhiều seed → độ nhiễu 2σ của val macro-F1; `CLIP_C` = trung vị grad_norm của `base-s1`.
3. **Các thí nghiệm** của 7 chủ đề, mỗi thí nghiệm đổi một yếu tố so với baseline (cùng split, 20 epoch, seed 1).
4. **Cấu hình cuối** chọn bằng val theo quy tắc trong `final:` của file YAML.

Mỗi lần chạy lưu `results/<exp_id>.json`, `figures/<exp_id>.png` và checkpoint `best_state` (ngoài thư mục nộp).
`run_experiment(cfg, data)` trong `train.py` là hàm huấn luyện duy nhất; `runner.py` chỉ điều phối.

In [ ]:
PLAN = load_plan(CODE_DIR / CONFIG_FILE)
runner = Runner(PLAN, data, FIG_DIR, RES_DIR, CKPT_DIR, skip_existing=SKIP_EXISTING,
                epochs_override=2 if QUICK else None)
print(f"{len(PLAN['lr_search']['lrs'])} lr search + {len(PLAN['base_seeds'])} baseline + "
      f"{len(PLAN['experiments'])} thí nghiệm + {len(PLAN['final']['seeds'])} final")
runner.run_all(device)

RESULTS, base_ids, final_ids, NOISE = runner.results, runner.base_ids, runner.final_ids, runner.noise
BASE_LR, CLIP_C = runner.ctx["BASE_LR"], runner.ctx["CLIP_C"]
if runner.skipped:
    print("Bỏ qua:", runner.skipped)

def show(patterns, compare=None):
    df = runner.table(patterns)
    display(df.style.format(precision=4, na_rep="—"))
    for name in ([compare] if isinstance(compare, str) else compare or []):
        p = f"{FIG_DIR}/compare_{name}.png"
        if os.path.exists(p):
            display_img(p)
    return df

### 2a. Chọn lr baseline bằng val

In [ ]:
show(["lr-sgdm-*"], "lr_sgdm")
print("BASE_LR =", BASE_LR)

### 2b. Baseline với nhiều seed → độ nhiễu

In [ ]:
f1s = np.array([RESULTS[e]["summary"]["val_macro_f1"] for e in base_ids])
accs = np.array([RESULTS[e]["summary"]["val_acc"] for e in base_ids])
print(f"val_macro_f1 = {f1s.mean():.4f} ± {f1s.std(ddof=1):.4f} (std mẫu, n={len(f1s)}) -> ngưỡng nhiễu 2σ = {NOISE['two_sigma']:.4f}")
print(f"val_acc      = {accs.mean():.4f} ± {accs.std(ddof=1):.4f}   (mốc 'đoán đa số' = 0.4876)")
show(base_ids, "baseline")
display_img(f"{FIG_DIR}/base-s1.png")

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số" 0.4876, độ nhiễu giữa các seed ...

## Part 3 — Kết quả từng chủ đề
So sánh với `base-s1` và với ngưỡng nhiễu 2σ của val macro-F1 (cột `vượt 2σ?`).

### Chủ đề 1 — Hàm mất mát: CE vs MSE (trên one-hot)
MSE = trung bình bình phương trên mọi phần tử `(B × 7)` giữa logits và one-hot (như `nn.MSELoss`, không có hệ số 1/2).
Chạy ở lr baseline và lr ×10. Không so loss trực tiếp (khác thang đo); so accuracy / macro-F1 / tốc độ hội tụ.

**Dự đoán trước (tự viết):** ...

In [ ]:
show(["base-s1", "loss-*"], "loss")

**Đối chiếu (tự viết):** ...

### Chủ đề 2 — Bộ tối ưu hoá
SGD, SGD+momentum (lưới lr ở 2a), Adam, AdamW (`weight_decay=0.01`), mỗi bộ ≥ 3 lr; so ở **lr tốt nhất của mỗi bộ**.
Tham số: momentum 0.9; Adam/AdamW `betas=(0.9, 0.999)`, `eps=1e-8`.

**Dự đoán trước (tự viết):** ...

In [ ]:
show(["lr-sgdm-*", "opt-*"])
print("lr tốt nhất của mỗi bộ tối ưu (theo val macro-F1):")
show(["best:lr-sgdm-*", "best:opt-sgd-*", "best:opt-adam-*", "best:opt-adamw-*"], "optimizer")

**Đối chiếu (tự viết):** ...

### Chủ đề 3 — Hyper-parameter
Batch 128 / 2048 (cùng lr → số bước cập nhật khác nhau), batch 2048 + lr ×4 (quy tắc tăng lr theo lô — đổi 2 yếu tố, có ghi rõ),
`M-wide`, `M-deep`, weight decay 5e-4, cosine lr schedule.

**Dự đoán trước (tự viết):** ...

In [ ]:
show(["base-s1", "hp-*"], ["batch", "hparam"])

**Đối chiếu (tự viết):** ...

### Chủ đề 4 — Dropout
q ∈ {0.1, 0.3, 0.5}, sau ReLU của mỗi lớp ẩn. Train loss đo ở `eval()` nên so được với val loss.

**Dự đoán trước (tự viết):** (gợi ý: baseline có đang quá khớp không? nhìn khoảng cách train–val của `base-s1`)

In [ ]:
df = show(["base-s1", "drop-*"], "dropout")
display((df["final_val"] - df["final_train"]).rename("gap val - train (epoch cuối)").to_frame())

**Đối chiếu (tự viết):** ...

### Chủ đề 5 — Gradient clipping
`c = CLIP_C` = trung vị grad norm (TB theo epoch) của `base-s1`, để clipping thực sự kích hoạt ở một phần các bước.
Thí nghiệm phản chứng: lr ×10 **không clip** vs **có clip**.

**Dự đoán trước (tự viết):** ...

In [ ]:
print("grad_norm TB theo epoch của base-s1:", np.round(RESULTS["base-s1"]["history"]["grad_norm"], 3))
print(f"CLIP_C = {CLIP_C}, lr cao = {BASE_LR * 10:g}")
show(["base-s1", "clip-*", "noclip-*"], "clipping")
for e in runner.match(["clip-*"]):
    print(f"{e}: tỉ lệ bước bị cắt theo epoch = {np.round(RESULTS[e]['history']['clip_frac'], 2)}")

**Đối chiếu (tự viết):** ...

### Chủ đề 6 — Mixed precision
FP16: `autocast(float16)` + `GradScaler` (luôn `unscale_` trước khi đo/cắt grad norm). BF16: `autocast(bfloat16)`, không GradScaler.
Tham số vẫn FP32. Đo thời gian/epoch, bộ nhớ cực đại, độ chính xác so với FP32.

**Dự đoán trước (tự viết):** ...

In [ ]:
print("bỏ qua:", {k: v for k, v in runner.skipped.items() if k.startswith("amp")} or "không")
show(["base-s1", "amp-*"], "amp")

**Đối chiếu (tự viết):** ...

### Chủ đề 7 — Khởi tạo tham số
`zeros`, `normal` (std 0.01), `xavier` (`xavier_normal_`: Var = 2/(n_in+n_out)), `he` (baseline: Var = 2/n_in), `default` (mặc định
`nn.Linear`: kaiming_uniform với a=√5, Var ≈ 1/(3·n_in)). Bias = 0 (trừ `default`). Std kích hoạt sau mỗi ReLU ở bước 0 (một lô val)
cho M-base và cho một mạng sâu 20 lớp ẩn 256 (chỉ để quan sát, không huấn luyện).

**Dự đoán trước (tự viết):** (gợi ý: với `zeros`, gradient của các nơ-ron cùng lớp ra sao? ReLU(0) = 0 ...)

In [ ]:
xb_val = data["X_val"][:4096]
act_rows, deep_rows = {}, {}
for init in ("zeros", "normal", "xavier", "he", "default"):
    set_seed(1)
    act_rows[init] = activation_stats(MLP(hidden=(256, 128), init=init).to(device), xb_val)
    set_seed(1)
    deep_rows[init] = activation_stats(MLP(hidden=(256,) * 20, init=init).to(device), xb_val)
display(pd.DataFrame(act_rows, index=["ReLU 1", "ReLU 2", "logits"]).T.style.format("{:.4g}")
        .set_caption("std kích hoạt ở bước 0, M-base"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for init, stds in act_rows.items():
    axes[0].plot(range(1, len(stds) + 1), np.maximum(stds, 1e-12), "o-", label=init)
for init, stds in deep_rows.items():
    axes[1].plot(range(1, len(stds) + 1), np.maximum(stds, 1e-12), "o-", ms=3, label=init)
for ax, t in zip(axes, ["M-base (2 lớp ẩn + logits)", "Mạng 20 lớp ẩn 256 (không huấn luyện)"]):
    ax.set(yscale="log", xlabel="lớp (sau ReLU; điểm cuối = logits)", ylabel="std kích hoạt", title=t)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle("Std kích hoạt ở bước 0 theo cách khởi tạo (giá trị 0 được vẽ ở 1e-12)")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/compare_init_activations.png", dpi=110, bbox_inches="tight"); plt.show()

show(["base-s1", "init-*"], "init")

**Đối chiếu (tự viết):** ...

### Cấu hình cuối (chọn CHỈ bằng val)
Quy tắc (mục `final:` trong `experiments.yaml`): optimizer/lr lấy từ thí nghiệm có val macro-F1 cao nhất trong lưới lr + nhóm optimizer;
kiến trúc / cosine / dropout chỉ thêm nếu thí nghiệm đơn lẻ của nó **vượt baseline hơn 2σ**; huấn luyện `final.epochs` epoch,
giữ epoch có val loss thấp nhất. File nộp là seed đầu tiên.

In [ ]:
print(runner.final_desc)
c = RESULTS[final_ids[0]]["cfg"]
print({k: c[k] for k in ("optimizer", "lr", "weight_decay", "hidden", "dropout", "scheduler", "epochs", "batch")})
ff = np.array([RESULTS[e]["summary"]["val_macro_f1"] for e in final_ids])
print(f"final val_macro_f1 = {ff.mean():.4f}" + (f" ± {ff.std(ddof=1):.4f}" if len(ff) > 1 else "") +
      f"  vs baseline {NOISE['mean']:.4f} ± {NOISE['std']:.4f}")
show(base_ids + final_ids, "final")

**Nhận xét cấu hình cuối (tự viết):** ...

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
Cấu hình đã chọn ở trên **chỉ bằng val**. Giờ mới dự đoán trên toàn bộ eval (eval mode, trọng số ở epoch có val loss thấp nhất)
và chấm bằng đúng `scripts/evaluate.py`. Chỉ làm cho baseline (`base-s1`) và cấu hình cuối (`final-s<seed đầu>`).

In [ ]:
SUBMIT_ID = final_ids[0]
os.makedirs(f"{OUT_DIR}/eval_baseline", exist_ok=True)
print("=== Baseline (base-s1) ===")
EVAL_BASE = final_eval(RESULTS["base-s1"]["cfg"], RESULTS["base-s1"], data,
                       f"{OUT_DIR}/eval_baseline/predictions_baseline.csv", repo_root=str(REPO_ROOT),
                       out_json=f"{OUT_DIR}/eval_baseline/eval_result_baseline.json")
print(f"=== Cấu hình cuối ({SUBMIT_ID}) -> file nộp ===")
EVAL_FINAL = final_eval(RESULTS[SUBMIT_ID]["cfg"], RESULTS[SUBMIT_ID], data,
                        f"{OUT_DIR}/predictions_eval.csv", repo_root=str(REPO_ROOT),
                        out_json=f"{OUT_DIR}/eval_result.json")
print(f"baseline: eval acc {EVAL_BASE['accuracy']:.4f}, macro-F1 {EVAL_BASE['macro_f1']:.4f}")
print(f"final   : eval acc {EVAL_FINAL['accuracy']:.4f}, macro-F1 {EVAL_FINAL['macro_f1']:.4f}  "
      f"(Δ = {EVAL_FINAL['macro_f1'] - EVAL_BASE['macro_f1']:+.4f})")
for e, ev in [("base-s1", EVAL_BASE), (SUBMIT_ID, EVAL_FINAL)]:
    print(f"{e}: val F1 {RESULTS[e]['summary']['val_macro_f1']:.4f} vs eval F1 {ev['macro_f1']:.4f}")

In [ ]:
# Phân tích lỗi theo lớp trên eval
n_train_cls = np.bincount(data["y_tr"].cpu().numpy(), minlength=7)
pc = pd.DataFrame(EVAL_FINAL["per_class"]).set_index("cls")
pc["train_count"] = n_train_cls
pc["f1_baseline"] = [c["f1"] for c in EVAL_BASE["per_class"]]
pc["Δf1"] = pc["f1"] - pc["f1_baseline"]
display(pc.style.format({"precision": "{:.4f}", "recall": "{:.4f}", "f1": "{:.4f}", "f1_baseline": "{:.4f}", "Δf1": "{:+.4f}"})
        .set_caption(f"Theo lớp trên eval — {SUBMIT_ID} (nhãn gốc = cls + 1)"))

cm = np.array(EVAL_FINAL["confusion_matrix"])
cm_norm = cm / cm.sum(1, keepdims=True)
display(pd.DataFrame(cm, index=[f"thật {c}" for c in range(7)], columns=[f"đoán {c}" for c in range(7)]))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set(xticks=range(7), yticks=range(7), xlabel="dự đoán", ylabel="nhãn thật",
       title=f"Ma trận nhầm lẫn chuẩn hoá theo hàng (recall) — eval, {SUBMIT_ID}")
fig.colorbar(im); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
hardest = int(pc["f1"].idxmin())
print(f"lớp F1 thấp nhất: {hardest} (F1 {pc.loc[hardest, 'f1']:.4f}, {n_train_cls[hardest]:,} mẫu train); "
      f"hay bị đoán nhầm thành lớp {int(off[hardest].argmax())} ({off[hardest].max()} / {cm[hardest].sum()} mẫu)")
top = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:5], off.shape))[0]
print("5 cặp nhầm nhiều nhất (thật -> đoán: số mẫu):", [(int(i), int(j), int(off[i, j])) for i, j in top])

**Phân tích lỗi (tự viết):** lớp khó nhất ..., nhầm với lớp ..., vì sao (số mẫu, đặc trưng tương đồng) ...

In [ ]:
# Bảng experiments.xlsx: mỗi exp_id đã chạy là một dòng (theo thứ tự chạy); eval chỉ cho base-s1 và cấu hình cuối
eval_of = {"base-s1": EVAL_BASE, SUBMIT_ID: EVAL_FINAL}
note_of = {"base-s1": "baseline dùng để chấm eval", SUBMIT_ID: "cấu hình cuối, file predictions_eval.csv"}
rows = [to_row(r, eval_scores=eval_of.get(e), notes=note_of.get(e, "")) for e, r in RESULTS.items()]
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx", seed_ids=base_ids)
display(pd.DataFrame(rows).set_index("exp_id")[["group", "optimizer", "lr", "val_acc", "val_macro_f1", "eval_acc", "eval_macro_f1", "diverged"]])
print(f"đã ghi {OUT_DIR}/experiments.xlsx ({len(rows)} dòng) — mở bằng Excel/LibreOffice để các cột công thức tự tính")

In [ ]:
# Kiểm tra nhanh trước khi nộp (README mục 6.4)
figs = {p.stem for p in Path(FIG_DIR).glob("*.png") if not p.stem.startswith("compare_")}
ids = {r["exp_id"] for r in rows}
print("số dòng bảng:", len(ids), "| số ảnh figures/<exp_id>.png:", len(figs))
assert figs == ids, f"thiếu ảnh: {ids - figs}, ảnh thừa: {figs - ids}"
for f in ["experiments.xlsx", "predictions_eval.csv", "eval_result.json", "code/lab.ipynb"]:
    assert (SUB_DIR / f).exists(), f
pred = pd.read_csv(SUB_DIR / "predictions_eval.csv")
assert list(pred.columns) == ["row_id", "pred"] and len(pred) == 116_203 and pred["row_id"].is_unique
leftover = [p.name for p in CODE_DIR.glob("*.py") if "NotImplementedError" in p.read_text(encoding="utf-8")]
assert not leftover, leftover
print("OK. Còn lại phần tự làm: viết REPORT.md, nhận xét trong notebook và sheet Summary; lưu notebook (có output) vào",
      CODE_DIR / "lab.ipynb")
print("Lưu ý: không nộp thư mục eval_baseline/ nếu không cần; checkpoints nằm ở", CKPT_DIR, "(ngoài thư mục nộp)")